In [13]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import pydicom
from pydicom.data import get_testdata_files
import glob
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ==================== 路径配置 ====================
BASE_ROOT = Path.cwd().parent  # 当前工作目录，你也可以改成绝对路径
DATA_ROOT = BASE_ROOT / "data" / "Task09_Spleen" / "Task09_Spleen"  # MSD 数据集根目录
OUTPUT_DIR = BASE_ROOT / "outputs" / "m3_task1_dicom_nifti"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [14]:
# 获取测试文件
dcm_files = get_testdata_files("CT_small.dcm")
if not dcm_files:
    raise FileNotFoundError("无法获取 CT_small.dcm 测试文件")
dcm_path = dcm_files[0]
print(f"DICOM 文件路径: {dcm_path}")

# 读取 DICOM
ds = pydicom.dcmread(dcm_path)
pixel_array = ds.pixel_array  # 2D 单切片

DICOM 文件路径: D:\miniconda3\envs\cv_base\Lib\site-packages\pydicom\data\test_files\CT_small.dcm


In [17]:
ds.Columns

128

In [15]:
pixel_array.shape

(128, 128)

In [9]:
image_dir = DATA_ROOT / "imagesTr"
label_dir = DATA_ROOT / "labelsTr"
image_files = sorted(glob.glob(str(image_dir / "*.nii.gz")))
label_files = sorted(glob.glob(str(label_dir / "*.nii.gz")))
seleted_image = image_files[0]
seleted_label = label_files[0]
print(seleted_image)

F:\PythonProjects\cv_study_project\data\Task09_Spleen\Task09_Spleen\imagesTr\spleen_10.nii.gz


In [19]:
# 读取图像和标签
img_nib = nib.load(seleted_image)
lbl_nib = nib.load(seleted_label)
# 获取像素数据 (nibabel 默认是 (X, Y, Z)，我们转置为 (Z, Y, X) 方便切片)
image_data = img_nib.get_fdata().astype(np.float32)
label_data = lbl_nib.get_fdata().astype(np.uint8)

# 转置为 (Z, Y, X)
image_data = np.transpose(image_data, (2, 1, 0))
label_data = np.transpose(label_data, (2, 1, 0))

# 提取空间信息
affine = img_nib.affine
header = img_nib.header
pixdim = header.get_zooms()  # (X, Y, Z) 方向的 spacing
# 转置后 spacing 对应 (Z, Y, X)
spacing = (pixdim[2], pixdim[1], pixdim[0])  # (Z, Y, X)

# 提取 origin 和 direction（从 affine 矩阵计算）
origin = affine[:3, 3]  # 物理原点
# direction cosines (旋转矩阵)
direction = affine[:3, :3]